# AlphaFold DB Fetch — Example

![AlphaFold DB Fetch — Example](https://proto-bio.github.io/proto-assets/images/tool/alphafold_db/hero.png)

This notebook demonstrates `run_alphafold_db_fetch`, which retrieves an AlphaFold prediction record from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) by UniProt accession. It returns the predicted structure (PDB or mmCIF), per-residue pLDDT confidence scores, and optionally the predicted aligned error (PAE) matrix, alongside metadata such as gene name, organism, model version, and source URLs. See the AlphaFold DB paper: Varadi et al., *Nucleic Acids Research* (2022), [doi:10.1093/nar/gkab1061](https://doi.org/10.1093/nar/gkab1061).

In [ ]:
from proto_tools.tools.database_retrieval import (
    AlphaFoldDBFetchConfig,
    AlphaFoldDBFetchInput,
    run_alphafold_db_fetch,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

In [ ]:
display_api_reference("alphafold_db", "input")

## Config API Reference

In [ ]:
display_api_reference("alphafold_db", "config")

## Output API Reference

In [ ]:
display_api_reference("alphafold_db", "output")

In [ ]:
# Basic usage: fetch the AlphaFold prediction for human TP53 (UniProt P04637)
# with all defaults (PDB structure + per-residue pLDDT, PAE off).
inputs = AlphaFoldDBFetchInput(uniprot_id="P04637")
config = AlphaFoldDBFetchConfig()

output = run_alphafold_db_fetch(inputs, config)

print(f"UniProt accession: {output.uniprot_accession}")
print(f"AlphaFold entry id: {output.entry_id}")
print(f"Sequence length:   {output.sequence_length}")
print(f"Mean pLDDT:        {output.mean_plddt}")

print(f"\nStructure format:  {output.structure.structure_format}")
print(f"B-factor type:     {output.structure.b_factor_type.value}")
print("\nStructure file (first 80 chars):")
print(output.structure.structure[:80])

plddt = output.structure.metrics["plddt_per_residue"]
print(f"\nstructure.metrics keys:   {sorted(output.structure.metrics.keys())}")
print(f"per-residue pLDDT length: {len(plddt)}")
print(f"first 5 pLDDT values:     {plddt[:5]}")

In [ ]:
# With the PAE matrix: useful for assessing inter-domain confidence.
# PAE files can be large for long proteins, so this is opt-in.
pae_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_pae=True),
)

pae = pae_output.structure.metrics["pae"]
print(f"PAE matrix shape: ({len(pae)}, {len(pae[0])})")

print("\nTop-left 3x3 corner (angstroms):")
for row in pae[:3]:
    print([round(v, 3) for v in row[:3]])

In [ ]:
# Metadata-only fetch: skip the structure download.
# This is the cheapest call when you only need URLs and identifiers.
meta_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_structure=False),
)

print(f"PDB URL:        {meta_output.pdb_url}")
print(f"mmCIF URL:      {meta_output.cif_url}")
print(f"pLDDT doc URL:  {meta_output.plddt_doc_url}")
print(f"PAE doc URL:    {meta_output.pae_doc_url}")
print(f"Mean pLDDT:     {meta_output.mean_plddt}")
print(f"structure is None: {meta_output.structure is None}")

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("alphafold_db_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'alphafold_db_results.json'}")